# Predicción de Binding Pockets con Graph Neural Networks

Pipeline completo para predecir sitios de unión fármaco-proteína (binding pockets) usando GNNs entrenadas en ~500-1000 estructuras PDB reales.

## Flujo general:
1. Descarga de estructuras desde RCSB PDB (~500-1000 complejos proteína-ligando)
2. Parseo: extrae residuos (C-alfa) y ligandos, calcula features (SASA, hidrofobicidad, carga, densidad)
3. Construcción de grafos: nodos=residuos, edges=contactos geométricos (≤8Å), features=química+geometría
4. Entrenamiento: GNN con message passing consciente de distancias + Focal Loss para manejar desbalance (~1:30-100)
5. Visualización 3D: heatmaps de predicción vs. ground truth (TP/FP/FN en colores)


## 0. Instalación de dependencias

In [1]:
# Descomenta si estás en Colab o entorno limpio
# !pip install biopython scipy pandas numpy matplotlib seaborn scikit-learn tqdm requests -q
# !pip install torch --index-url https://download.pytorch.org/whl/cu121 -q
# !pip install torch_geometric -q
# !pip install py3Dmol -q


In [2]:
import os
import glob
import urllib.request
import json
import requests
from typing import Tuple, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
from scipy.spatial.distance import cdist
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.norm import BatchNorm

from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


c:\Users\angel\miniconda3\envs\dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\angel\miniconda3\envs\dev\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] No se encontró el proceso especificado
  import torch_geometric.typing
c:\Users\angel\miniconda3\envs\dev\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] No se encontró el proceso especificado
  import torch_geometric.typing


Device: cpu


## 1. Configuración global y constantes

In [3]:
RAW_DIR = "./data/raw_pdb"
PROCESSED_DIR = "./data/processed_graphs"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Mapeo de aminoácidos a índices (para one-hot)
AA_MAP = {
    "ALA": 0, "ARG": 1, "ASN": 2, "ASP": 3, "CYS": 4,
    "GLN": 5, "GLU": 6, "GLY": 7, "HIS": 8, "ILE": 9,
    "LEU": 10, "LYS": 11, "MET": 12, "PHE": 13, "PRO": 14,
    "SER": 15, "THR": 16, "TRP": 17, "TYR": 18, "VAL": 19,
}
NUM_AA_CLASSES = len(AA_MAP) + 1  # +1 para desconocidos

# Hetatm a ignorar (agua, iones, buffers — no son ligandos de interés)
IGNORED_HETATM = {
    "HOH", "WAT", "NA", "CL", "MG", "ZN", "CA", "SO4", "PO4",
    "EDO", "DMS", "ACT", "GOL", "PEG", "MPD",
}

# Propiedades fisicoquímicas por aminoácido
KD_HYDROPHOBICITY = {
    "ALA": 1.8, "ARG": -4.5, "ASN": -3.5, "ASP": -3.5, "CYS": 2.5,
    "GLN": -3.5, "GLU": -3.5, "GLY": -0.4, "HIS": -3.2, "ILE": 4.5,
    "LEU": 3.8, "LYS": -3.9, "MET": 1.9, "PHE": 2.8, "PRO": -1.6,
    "SER": -0.8, "THR": -0.7, "TRP": -0.9, "TYR": -1.3, "VAL": 4.2,
}

CHARGE = {
    "ARG": 1.0, "LYS": 1.0, "HIS": 0.1,
    "ASP": -1.0, "GLU": -1.0,
}

# SASA máxima teórica por residuo (para normalización)
MAX_ASA = {
    "ALA": 129, "ARG": 274, "ASN": 195, "ASP": 193, "CYS": 167,
    "GLN": 225, "GLU": 223, "GLY": 104, "HIS": 224, "ILE": 197,
    "LEU": 201, "LYS": 236, "MET": 224, "PHE": 240, "PRO": 159,
    "SER": 155, "THR": 172, "TRP": 285, "TYR": 263, "VAL": 174,
}

EXTRA_FEATURE_DIM = 3  # hidrofobicidad, carga, SASA relativa

sasa_calculator = ShrakeRupley()


## 2. Descarga de PDBs desde RCSB

Ampliar dataset consultando la RCSB Search API por entradas con proteína + ligando + buena resolución.


In [4]:
def fetch_pdb_ids_rcsb(max_results: int = 800, max_resolution: float = 2.5) -> List[str]:
    """
    Consulta RCSB Search API por complejos con:
    - Al menos 1 cadena de proteína
    - Al menos 1 entidad no-polimérica (ligando)
    - Resolución <= max_resolution (Å)
    Devuelve lista de PDB IDs ordenados por resolución ascendente.
    """
    query = {
        "query": {
            "type": "group",
            "logical_operator": "and",
            "nodes": [
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.polymer_entity_count_protein",
                        "operator": "greater_or_equal",
                        "value": 1
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.nonpolymer_entity_count",
                        "operator": "greater_or_equal",
                        "value": 1
                    }
                },
                {
                    "type": "terminal",
                    "service": "text",
                    "parameters": {
                        "attribute": "rcsb_entry_info.resolution_combined",
                        "operator": "less_or_equal",
                        "value": max_resolution
                    }
                }
            ]
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {"start": 0, "rows": max_results},
            "sort": [{"sort_by": "rcsb_entry_info.resolution_combined", "direction": "asc"}]
        }
    }

    url = "https://search.rcsb.org/rcsbsearch/v2/query"
    resp = requests.post(url, data=json.dumps(query), headers={"Content-Type": "application/json"})
    resp.raise_for_status()
    results = resp.json()
    return [hit["identifier"] for hit in results.get("result_set", [])]


def download_pdb(pdb_id: str, dest_dir: str = RAW_DIR) -> Optional[str]:
    """Descarga un .pdb desde RCSB si no existe ya localmente."""
    pdb_id = pdb_id.lower()
    file_path = os.path.join(dest_dir, f"{pdb_id}.pdb")
    if os.path.exists(file_path):
        return file_path
    try:
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
        urllib.request.urlretrieve(url, file_path)
        return file_path
    except Exception:
        return None


def download_all_parallel(pdb_ids: List[str], max_workers: int = 8) -> List[str]:
    """
    Descarga en paralelo con pool acotado (respeta límites de RCSB).
    Reintenta los que fallen una vez.
    """
    paths = []
    failed = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(download_pdb, pid): pid for pid in pdb_ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Descargando PDBs"):
            pid = futures[future]
            try:
                path = future.result()
                if path is not None:
                    paths.append(path)
                else:
                    failed.append(pid)
            except Exception:
                failed.append(pid)

    if failed:
        print(f"\n{len(failed)} fallaron, reintentando...")
        for pid in tqdm(failed, desc="Reintentos"):
            path = download_pdb(pid)
            if path is not None:
                paths.append(path)

    return paths


# Generar lista de PDBs
print("Consultando RCSB Search API...")
new_ids = fetch_pdb_ids_rcsb(max_results=800, max_resolution=2.5)
print(f"Encontrados {len(new_ids)} PDB IDs")

# Descargar todos en paralelo
pdb_paths = download_all_parallel(new_ids, max_workers=8)
print(f"Descargados {len(pdb_paths)} / {len(new_ids)} archivos PDB")


Consultando RCSB Search API...
Encontrados 800 PDB IDs


Descargando PDBs: 100%|██████████| 800/800 [00:20<00:00, 38.20it/s]



32 fallaron, reintentando...


Reintentos: 100%|██████████| 32/32 [00:18<00:00,  1.69it/s]

Descargados 768 / 800 archivos PDB


## 3. Cálculo de SASA y features fisicoquímicas

SASA (Solvent Accessible Surface Area) se calcula para cada residuo usando el algoritmo de Shrake-Rupley (ya incluido en Biopython).


In [5]:
def compute_residue_sasa(structure) -> dict:
    """
    Calcula SASA por residuo usando Shrake-Rupley.
    Devuelve {(chain_id, res_id): sasa_absoluto_en_A2}.
    """
    sasa_calculator.compute(structure, level="R")
    residue_sasa = {}
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, res_id, _ = residue.get_id()
                if hetflag == " ":
                    residue_sasa[(chain.get_id(), res_id)] = residue.sasa
        break
    return residue_sasa


## 4. Construcción de grafos PyTorch Geometric

Convierte cada PDB en un objeto `Data`:
- **Nodos**: residuos (C-alfa) con features de química + geometría
- **Edges**: radio_graph con corte 8.0 Å (contactos espaciales)
- **edge_attr**: distancia euclidea entre residuos conectados
- **y**: etiqueta binaria (binding site sí/no)


In [6]:
import os
import shutil
import time
from concurrent.futures import ProcessPoolExecutor, TimeoutError
from typing import List, Optional
import numpy as np
import torch
import torch.nn.functional as F
from scipy.spatial import cKDTree, distance
from torch_geometric.data import Data
from tqdm import tqdm


def process_single_pdb(args):
    """Función de nivel superior requerida para ProcessPoolExecutor."""
    path, cutoff_radius, contact_threshold = args
    pdb_id = os.path.basename(path).split(".")[0]
    try:
        data = pdb_to_graph_data(
            path,
            cutoff_radius=cutoff_radius,
            contact_threshold=contact_threshold,
        )
    except Exception:
        return None
    if data is None:
        return None
    return (path, pdb_id, data)


def build_and_save_graphs_parallel(
    pdb_paths: List[str],
    out_dir: str = PROCESSED_DIR,
    cutoff_radius: float = 8.0,
    contact_threshold: float = 5.0,
    max_workers: int = 4,
    max_samples: Optional[int] = None,
    max_duration_seconds: float = 15 * 60,
    timeout_per_sample: float = 30.0,
) -> List[str]:
    """
    Construye y guarda grafos usando ProcessPoolExecutor con timeout por archivo
    y límite global estricto.
    """
    saved_paths = []
    skipped = 0
    start_time = time.time()

    executor = ProcessPoolExecutor(max_workers=max_workers)
    
    # Argumentos empaquetados para la llamada por proceso
    tasks = [(p, cutoff_radius, contact_threshold) for p in pdb_paths]
    futures = {executor.submit(process_single_pdb, task): task[0] for task in tasks}

    try:
        with tqdm(total=len(futures), desc="Construyendo grafos (procesos)") as pbar:
            for future in list(futures.keys()):
                elapsed_time = time.time() - start_time

                # Límite global de tiempo
                if max_duration_seconds is not None and elapsed_time >= max_duration_seconds:
                    print(f"\n[!] Límite de tiempo alcanzado ({elapsed_time:.1f}s). Abortando.")
                    break

                # Límite de muestras
                if max_samples is not None and len(saved_paths) >= max_samples:
                    print(f"\n[i] Límite de muestras alcanzado ({max_samples} grafos). Finalizando.")
                    break

                try:
                    # Timeout individual por muestra para evitar que un PDB congele el worker
                    result = future.result(timeout=timeout_per_sample)
                    if result is not None:
                        path, pdb_id, data = result
                        out_path = os.path.join(out_dir, f"{pdb_id}.pt")
                        torch.save(data, out_path)
                        saved_paths.append(out_path)
                    else:
                        skipped += 1
                except TimeoutError:
                    skipped += 1
                    # Cancelar el futuro atascado
                    future.cancel()
                except Exception:
                    skipped += 1

                pbar.update(1)

    finally:
        # shutdown(wait=False, cancel_futures=True) fuerza la salida sin colgarse
        executor.shutdown(wait=False, cancel_futures=True)

    total_time = time.time() - start_time
    print(f"\nGrafos guardados: {len(saved_paths)} | Omitidos: {skipped} | Tiempo total: {total_time:.2f}s")
    return saved_paths

In [7]:
# Limpieza de directorio
shutil.rmtree(PROCESSED_DIR, ignore_errors=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Ejecución limitando a X muestras o máximo 15 minutos (900 s)
MAX_SAMPLES = 500  # Cambiar por tu valor X deseado (o None para no limitar por cantidad)
MAX_TIME_SEC = 10 * 60

graph_paths = build_and_save_graphs_parallel(
    pdb_paths=pdb_paths,
    out_dir=PROCESSED_DIR,
    max_workers=4,
    max_samples=MAX_SAMPLES,
    max_duration_seconds=MAX_TIME_SEC,
)

Construyendo grafos (procesos): 100%|██████████| 768/768 [00:00<00:00, 122016.12it/s]


Grafos guardados: 0 | Omitidos: 768 | Tiempo total: 0.11s


## 5. Arquitectura del modelo GNN

**DistanceAwareConv**: capa de message passing que incorpora distancias euclídeas explícitamente.

**BindingSiteGNN**: stack de capas con LayerNorm (no BatchNorm) + head de clasificación binaria.


In [8]:
class DistanceAwareConv(MessagePassing):
    """
    Message passing que concatena [x_i, x_j, edge_attr(distancia)]
    en un MLP para generar el mensaje, luego agrega (mean) y actualiza.
    Inspirado en EGNN pero sin actualizar posiciones (no las necesitamos).
    """
    def __init__(self, in_channels: int, out_channels: int, edge_dim: int = 1):
        super().__init__(aggr="mean")
        self.msg_mlp = nn.Sequential(
            nn.Linear(2 * in_channels + edge_dim, out_channels),
            nn.SiLU(),
            nn.Linear(out_channels, out_channels),
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(in_channels + out_channels, out_channels),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_i, x_j, edge_attr):
        return self.msg_mlp(torch.cat([x_i, x_j, edge_attr], dim=-1))

    def update(self, aggr_out, x):
        return self.update_mlp(torch.cat([x, aggr_out], dim=-1))


class BindingSiteGNN(nn.Module):
    """
    GNN de 2 capas con DistanceAwareConv.
    in_channels: dimensión de features de entrada (se infiere del dataset)
    hidden_channels: 64 (capacity moderado para evitar overfitting)
    num_layers: 2 (field receptivo ~16Å suficiente para bolsillos de unión)
    """
    def __init__(self, in_channels: int = 24, hidden_channels: int = 64,
                 num_layers: int = 2, edge_dim: int = 1, dropout: float = 0.3):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, hidden_channels)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(DistanceAwareConv(hidden_channels, hidden_channels, edge_dim))
            self.norms.append(nn.LayerNorm(hidden_channels))

        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x, edge_index, edge_attr):
        h = self.input_proj(x)
        for conv, norm in zip(self.convs, self.norms):
            h_res = h
            h = conv(h, edge_index, edge_attr)
            h = norm(h)
            h = F.silu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = h + h_res  # residual connection
        return self.classifier(h)


## 6. Focal Loss (manejo de desbalance de clases)

In [9]:
class FocalLoss(nn.Module):
    """
    Focal Loss para clases desbalanceadas.
    alpha: peso de la clase positiva (0.75 → 3x más importante que negativa)
    gamma: exponente de focusing (2.0 → ignore ejemplos fáciles)
    """
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_term = alpha_t * (1 - p_t) ** self.gamma
        return (focal_term * bce).mean()


## 7. Dataset, split y evaluación

In [10]:
def load_dataset(processed_dir: str = PROCESSED_DIR) -> List[Data]:
    files = sorted(glob.glob(f"{processed_dir}/*.pt"))
    return [torch.load(f, weights_only=False) for f in files]


def train_val_split(dataset: List[Data], val_frac: float = 0.15, seed: int = 42):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(dataset), generator=g)
    n_val = max(1, int(len(dataset) * val_frac))
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return [dataset[i] for i in train_idx], [dataset[i] for i in val_idx]


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr)
        all_logits.append(logits.cpu())
        all_targets.append(batch.y.cpu())

    logits = torch.cat(all_logits).squeeze(-1).numpy()
    targets = torch.cat(all_targets).squeeze(-1).numpy()
    probs = 1 / (1 + np.exp(-logits))

    pr_auc = average_precision_score(targets, probs)
    roc_auc = roc_auc_score(targets, probs)

    precisions, recalls, thresholds = precision_recall_curve(targets, probs)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    return {"pr_auc": pr_auc, "roc_auc": roc_auc, "best_f1": best_f1, "best_threshold": best_threshold}


## 8. Bucle de entrenamiento

Dataset de ~500-1000 PDBs, split 85% train / 15% val, entrenamiento hasta convergencia.


In [11]:
def train(processed_dir: str = PROCESSED_DIR, hidden_channels: int = 64, num_layers: int = 2,
          lr: float = 1e-3, weight_decay: float = 1e-4, batch_size: int = 8,
          epochs: int = 150, patience: int = 20, device: str = DEVICE,
          focal_alpha: float = 0.75, focal_gamma: float = 2.0):

    dataset = load_dataset(processed_dir)
    if len(dataset) < 3:
        raise ValueError(f"Solo hay {len(dataset)} grafos. Necesitas más PDBs.")

    train_set, val_set = train_val_split(dataset)
    print(f"Train: {len(train_set)} proteínas | Val: {len(val_set)} proteínas")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

    # Infiere dimensión de entrada del dataset (robusto a cambios de features)
    in_channels = dataset[0].x.shape[-1]
    print(f"Feature dimension: {in_channels}")

    model = BindingSiteGNN(in_channels=in_channels, hidden_channels=hidden_channels,
                            num_layers=num_layers).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
    criterion = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

    history = []
    best_pr_auc = 0.0
    epochs_no_improve = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = criterion(logits, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs

        avg_loss = total_loss / len(train_set)
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step(val_metrics["pr_auc"])
        history.append({"epoch": epoch, "train_loss": avg_loss, **val_metrics})

        print(f"Epoch {epoch:03d} | loss={avg_loss:.4f} | "
              f"PR-AUC={val_metrics['pr_auc']:.4f} | ROC-AUC={val_metrics['roc_auc']:.4f} | "
              f"F1={val_metrics['best_f1']:.4f} (thr={val_metrics['best_threshold']:.3f})")

        if val_metrics["pr_auc"] > best_pr_auc:
            best_pr_auc = val_metrics["pr_auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping en epoch {epoch} (mejor PR-AUC: {best_pr_auc:.4f})")
            break

    if best_state is not None:
        torch.save(best_state, "./best_binding_site_gnn.pt")
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_pr_auc, train_set, val_set


In [12]:
# Lanzar entrenamiento
model, history_df, best_pr_auc, train_set, val_set = train(
    epochs=150, batch_size=8, patience=20,
    hidden_channels=64, num_layers=2,
    focal_alpha=0.75, focal_gamma=2.0,
    weight_decay=1e-4,
)
print(f"\nMejor PR-AUC en validación: {best_pr_auc:.4f}")


ValueError: Solo hay 0 grafos. Necesitas más PDBs.

## 9. Curvas de entrenamiento

In [ ]:
if len(history_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", linewidth=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Focal Loss")
    axes[0].set_title("Pérdida de entrenamiento")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["pr_auc"], label="PR-AUC", linewidth=2)
    axes[1].plot(history_df["epoch"], history_df["roc_auc"], label="ROC-AUC", linewidth=2)
    axes[1].plot(history_df["epoch"], history_df["best_f1"], label="Best F1", linewidth=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Métricas de validación")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## 10. Diagnóstico: distribución de probabilidades predichas

Chequea si el modelo colapsa o tiene una distribución razonable.


In [ ]:
all_probs = []
model.eval()
with torch.no_grad():
    for data_i in val_set:
        data_i = data_i.to(DEVICE)
        logits = model(data_i.x, data_i.edge_index, data_i.edge_attr)
        probs = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
        all_probs.append(probs)

all_probs = np.concatenate(all_probs)
print(f"Media: {all_probs.mean():.4f} | Mediana: {np.median(all_probs):.4f} | Std: {all_probs.std():.4f}")
print(f"% de nodos con prob > 0.5: {(all_probs > 0.5).mean():.2%}")

plt.figure(figsize=(8, 5))
plt.hist(all_probs, bins=50, edgecolor='black', alpha=0.7)
try:
    viz_threshold = history_df["best_threshold"].iloc[-1]
except NameError:
    viz_threshold = 0.5
plt.axvline(viz_threshold, color="red", linestyle="--", linewidth=2, label=f"umbral={viz_threshold:.3f}")
plt.xlabel("Probabilidad predicha")
plt.ylabel("Frecuencia")
plt.title("Distribución de probabilidades — val set")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 11. Visualización 3D: predicción vs. ground truth

Renderiza la proteína coloreando cada residuo según su clasificación.


In [ ]:
try:
    import py3Dmol
except ImportError:
    print("py3Dmol no instalado. Descomenta: pip install py3Dmol")


In [ ]:
def get_residue_identifiers(pdb_path: str) -> List[Tuple[str, int]]:
    """
    Reproduce el mismo filtrado que pdb_to_graph_data para obtener
    (chain_id, res_id) en el mismo orden que los nodos del grafo.
    """
    parser = PDBParser(QUIET=True)
    pdb_id = os.path.basename(pdb_path).split(".")[0]
    structure = parser.get_structure(pdb_id, pdb_path)

    identifiers = []
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, _, _ = residue.get_id()
                if hetflag == " " and "CA" in residue:
                    identifiers.append((chain.get_id(), residue.get_id()[1]))
        break
    return identifiers


@torch.no_grad()
def visualize_prediction(model, data: Data, pdb_path: str, device: str = DEVICE,
                          threshold: float = 0.5, width: int = 800, height: int = 600):
    """
    Renderiza proteína 3D coloreando TP (rojo) / FP (naranja) / FN (azul).
    """
    model.eval()
    data = data.to(device)
    logits = model(data.x, data.edge_index, data.edge_attr)
    probs = torch.sigmoid(logits).squeeze(-1).cpu().numpy()
    y_true = data.y.squeeze(-1).cpu().numpy()
    y_pred = (probs >= threshold).astype(int)

    identifiers = get_residue_identifiers(pdb_path)
    assert len(identifiers) == len(y_true), (
        f"Desajuste nodo-residuo: {len(identifiers)} vs {len(y_true)}"
    )

    tp, fp, fn = [], [], []
    for (chain_id, res_id), yt, yp in zip(identifiers, y_true, y_pred):
        if yt == 1 and yp == 1:
            tp.append((chain_id, res_id))
        elif yt == 0 and yp == 1:
            fp.append((chain_id, res_id))
        elif yt == 1 and yp == 0:
            fn.append((chain_id, res_id))

    with open(pdb_path) as f:
        pdb_str = f.read()

    view = py3Dmol.view(width=width, height=height)
    view.addModel(pdb_str, "pdb")
    view.setStyle({}, {"cartoon": {"color": "lightgray"}})

    def add_highlight(residues, color):
        by_chain = {}
        for chain_id, res_id in residues:
            by_chain.setdefault(chain_id, []).append(res_id)
        for chain_id, resi_list in by_chain.items():
            view.addStyle(
                {"chain": chain_id, "resi": resi_list},
                {"cartoon": {"color": color}, "stick": {"color": color, "radius": 0.3}},
            )

    add_highlight(fn, "blue")
    add_highlight(fp, "orange")
    add_highlight(tp, "red")

    view.zoomTo()
    view.setBackgroundColor("white")

    print(f"PDB: {data.pdb_id} | TP={len(tp):3d}  FP={len(fp):3d}  FN={len(fn):3d}  "
          f"(umbral={threshold:.3f})")
    return view


In [ ]:
# Visualizar todas las proteínas del val set
try:
    viz_threshold = history_df["best_threshold"].iloc[-1]
except NameError:
    viz_threshold = 0.5

for data_i in val_set:
    pdb_path_i = os.path.join(RAW_DIR, f"{data_i.pdb_id}.pdb")
    if not os.path.exists(pdb_path_i):
        print(f"PDB no encontrado: {pdb_path_i}")
        continue
    try:
        v = visualize_prediction(model, data_i, pdb_path_i, device=DEVICE, threshold=viz_threshold)
        v.show()
    except Exception as e:
        print(f"Error visualizando {data_i.pdb_id}: {e}")


## 12. Inferencia en un PDB nuevo (no visto en train/val)

Carga el mejor modelo guardado y predice sobre una proteína completamente externa al dataset.


In [ ]:
# Descargar un PDB nuevo (que NO esté en train/val) para prueba
new_pdb_id = "1xyz"  # <-- cambia a un PDB real que no descargaste arriba
new_pdb_path = download_pdb(new_pdb_id)

if new_pdb_path is not None and os.path.exists(new_pdb_path):
    try:
        new_data = pdb_to_graph_data(new_pdb_path)
        if new_data is not None:
            new_data = new_data.to(DEVICE)
            v = visualize_prediction(model, new_data, new_pdb_path, device=DEVICE, threshold=viz_threshold)
            v.show()
        else:
            print(f"No se pudo construir grafo de {new_pdb_id}")
    except Exception as e:
        print(f"Error: {e}")
else:
    print(f"No se pudo descargar {new_pdb_id}")


## Resumen del modelo

### Features de entrada por nodo (residuo):
- **One-hot aminoácido** (21 dims): identidad del residuo
- **Hidrofobicidad** (1 dim): escala Kyte-Doolittle [-1, 1]
- **Carga** (1 dim): carga neta del residuo a pH fisiológico
- **SASA relativa** (1 dim): exposición al solvente [0, 1.5]
- **Total: 24 dimensiones**

### Arquitectura:
- **Input projection**: Linear(24 → 64)
- **2 capas DistanceAwareConv**: message passing + LayerNorm + residual
  - Incorpora distancias euclídeas explícitamente
  - Field receptivo: ~16 Å (suficiente para bolsillos de unión)
- **Classification head**: Linear(64→32) + SiLU + Dropout + Linear(32→1)

### Output:
- **Probabilidad por residuo** [0, 1] de ser binding site
- **Umbral óptimo** automático: busca el que maximiza F1 en validación
- **Visualización**: TP (rojo), FP (naranja), FN (azul), resto (gris)

### Training:
- **Loss**: Focal Loss (α=0.75, γ=2.0) para manejar desbalance ~1:30-100
- **Optimizer**: AdamW + ReduceLROnPlateau
- **Datos**: ~500-1000 complejos proteína-ligando (RCSB PDB)
- **Métricas**: PR-AUC (principal), ROC-AUC, F1-score

### Próximos pasos:
1. Ajustar `alpha` y `gamma` de Focal Loss si es necesario
2. Probar con más capas (3-4) si val PR-AUC no satura
3. Explorar más features geométricas (ej. densidad local)
4. Entrenamiento con PDBbind Core/Refined Set completo para mayor generalización
